# Data Collection — Job Postings via API

**Module 1 · Step 1 of 2**

Fetches the full `jobs.json` dataset from IBM Skills Network (27,005 records) once,  
then filters locally by **Location** and **Key Skills** to produce two Excel files.

> The S3 endpoint ignores query params and always returns all records —  
> local filtering is the only way to get accurate per-city / per-technology counts.

**Outputs:**
- `outputs/job-postings.xlsx` — job count by city
- `outputs/job-postings-technologies.xlsx` — job count by technology

---
## 1. Setup

In [ ]:
import requests
import pandas as pd
from pathlib import Path
from openpyxl import Workbook

JOBS_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud"
    "/IBM-DA0321EN-SkillsNetwork/labs/module%201"
    "/Accessing%20Data%20Using%20APIs/jobs.json"
)

LOCATIONS = [
    "Los Angeles", "New York", "San Francisco",
    "Washington DC", "Seattle", "Austin", "Detroit",
]

TECHNOLOGIES = [
    "C", "C#", "C++", "Java", "JavaScript", "Python",
    "Scala", "Oracle", "SQL Server", "MySQL Server", "PostgreSQL", "MongoDB",
]

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

LOCATION_FILE = OUTPUT_DIR / "job-postings.xlsx"
TECH_FILE     = OUTPUT_DIR / "job-postings-technologies.xlsx"

print("Setup complete.")

---
## 2. Fetch Dataset (single request)

In [ ]:
response = requests.get(JOBS_URL)
response.raise_for_status()
all_jobs = response.json()

print(f"Total records fetched: {len(all_jobs):,}")
# Inspect keys of first record
print("Sample keys:", list(all_jobs[0].keys()))

---
## 3. Count Job Postings by Location

In [ ]:
def count_jobs_by_location(jobs, location):
    """Count records whose 'Location' field contains the given city (case-insensitive)."""
    return sum(
        1 for job in jobs
        if location.lower() in job.get("Location", "").lower()
    )

location_rows = [
    {"Location": loc, "Number of Jobs": count_jobs_by_location(all_jobs, loc)}
    for loc in LOCATIONS
]

df_locations = pd.DataFrame(location_rows)
df_locations

In [ ]:
# Save to Excel
df_locations.to_excel(LOCATION_FILE, index=False)
print(f"Saved: {LOCATION_FILE}")

---
## 4. Count Job Postings by Technology

In [ ]:
def count_jobs_by_technology(jobs, technology):
    """Count records whose 'Key Skills' field contains the technology (case-insensitive)."""
    return sum(
        1 for job in jobs
        if technology.lower() in job.get("Key Skills", "").lower()
    )

tech_rows = [
    {"Technology": tech, "Number of Jobs": count_jobs_by_technology(all_jobs, tech)}
    for tech in TECHNOLOGIES
]

df_tech = pd.DataFrame(tech_rows)
df_tech

In [ ]:
# Save to Excel
df_tech.to_excel(TECH_FILE, index=False)
print(f"Saved: {TECH_FILE}")